In [1]:
import os.path
import base64
import re
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

SCOPES = ['https://www.googleapis.com/auth/gmail.readonly']

def get_removal_links():
    creds = None
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)
    
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
            creds = flow.run_local_server(port=0)
        with open('token.json', 'w') as token:
            token.write(creds.to_json())

    service = build('gmail', 'v1', credentials=creds)

    # Search specifically for the "disavow" type emails
    query = "from:accounts.google.com" 
    results = service.users().messages().list(userId='me', q=query).execute()
    messages = results.get('messages', [])

    removal_links = []

    if not messages:
        print("No recovery removal emails found.")
        return []
    else:
        for msg in messages:
            message = service.users().messages().get(userId='me', id=msg['id'], format='full').execute()
            
            # Parts of the email (handling multi-part/mime)
            parts = message.get('payload').get('parts')
            body = ""

            if parts:
                for part in parts:
                    if part.get('mimeType') == 'text/plain':
                        body = base64.urlsafe_b64decode(part.get('body').get('data')).decode()
            else:
                body = base64.urlsafe_b64decode(message.get('payload').get('body').get('data')).decode()

            # Regex to find the specific Google Account Disavow URL
            links = re.findall(r'https://accounts\.google\.com/AccountDisavow\?[\w=&-]+', body)
            if links:
                removal_links.extend(links)

        # Deduplicate links
        removal_links = list(set(removal_links))
        
        print(f"Successfully collected {len(removal_links)} removal links.")
        return removal_links


In [2]:
get_removal_links()

Successfully collected 106 removal links.


['https://accounts.google.com/AccountDisavow?adt=AOX8kiotjF3wUW-lTJsg80sgxNEGVS73BS2HvWbJgv-hGuvK0H05beywoKV9NZ69&rfn=354',
 'https://accounts.google.com/AccountDisavow?adt=AOX8kioe63E0Zxn7eI8r6xf0gOvf-ZNW3FiON1PKqGXKwPjH83w2_oxr-dkjU2PR&rfn=354',
 'https://accounts.google.com/AccountDisavow?adt=AOX8kip1_c6YANon3TtL6EE9xC0chD0t-DXRRzGhHHUTKtCeZdUUEFi2wwXX1fXx&rfn=354',
 'https://accounts.google.com/AccountDisavow?adt=AOX8kioZ177HcB2veVikQCJhU4Mk-oFXbvQI6uU36uNfrjIGyxHHGR7knAFVq-Ik&rfn=2&anexp=-signup&hl=ru',
 'https://accounts.google.com/AccountDisavow?adt=AOX8kiqgKe8-9ZhR0av5U5yPAiRBsA1l7TvRe75IhzrbQuYhtbdMNjXAG9fh2X2_&rfn=354',
 'https://accounts.google.com/AccountDisavow?adt=AOX8kiqz8qMbhixevvl2s_gjVRnFUrDcLtYGE7sHjD0flXYM850kPmCe7LHG0mGF&rfn=354',
 'https://accounts.google.com/AccountDisavow?adt=AOX8kioqFdrX9inIh-FKNC1JXR4h9HEM4chDK2dzxhkp66PvwR_y5G8NWAIv4ZSU&rfn=354',
 'https://accounts.google.com/AccountDisavow?adt=AOX8kirftPBs4de54OvkX8fqdngd5aoZnypRJtM-sf30HYGArz9Z75p6YHhppcY8&

In [3]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import time

driver = webdriver.Chrome()

try:
    links = get_removal_links()
    print(f"Total links to process: {len(links)}")

    for link in links:
        print(f"Opening: {link}")
        driver.get(link)
        
        try:
            # 1. Wait for the button (shortened to 5 seconds for faster skipping)
            wait = WebDriverWait(driver, 5)
            
            # 2. Check if the "Remove" button exists
            remove_button = wait.until(EC.element_to_be_clickable((By.XPATH, "//button[contains(., 'Remove')]")))
            
            # 3. If found, click it
            remove_button.click()
            print("Action performed successfully.")
            time.sleep(2)
            
        except TimeoutException:
            # This triggers if the button doesn't appear within 5 seconds
            print(f"SKIPPING: 'Remove' button not found (link may be expired or already processed).")
            continue 
            
        except Exception as inner_e:
            print(f"SKIPPING: Unexpected error on this page: {inner_e}")
            continue
    
    print("\nBatch processing complete.")

except Exception as e:
    print(f"Critical error: {e}")

finally:
    driver.quit()

Successfully collected 106 removal links.
Total links to process: 106
Opening: https://accounts.google.com/AccountDisavow?adt=AOX8kiotjF3wUW-lTJsg80sgxNEGVS73BS2HvWbJgv-hGuvK0H05beywoKV9NZ69&rfn=354
Action performed successfully.
Opening: https://accounts.google.com/AccountDisavow?adt=AOX8kioe63E0Zxn7eI8r6xf0gOvf-ZNW3FiON1PKqGXKwPjH83w2_oxr-dkjU2PR&rfn=354
Action performed successfully.
Opening: https://accounts.google.com/AccountDisavow?adt=AOX8kip1_c6YANon3TtL6EE9xC0chD0t-DXRRzGhHHUTKtCeZdUUEFi2wwXX1fXx&rfn=354
Action performed successfully.
Opening: https://accounts.google.com/AccountDisavow?adt=AOX8kioZ177HcB2veVikQCJhU4Mk-oFXbvQI6uU36uNfrjIGyxHHGR7knAFVq-Ik&rfn=2&anexp=-signup&hl=ru
SKIPPING: 'Remove' button not found (link may be expired or already processed).
Opening: https://accounts.google.com/AccountDisavow?adt=AOX8kiqgKe8-9ZhR0av5U5yPAiRBsA1l7TvRe75IhzrbQuYhtbdMNjXAG9fh2X2_&rfn=354
Action performed successfully.
Opening: https://accounts.google.com/AccountDisavow?adt=AOX8kiq